### Data ingestion for old IPC , IEA and CCP acts.

In [1]:
import os
import re
import fitz
import numpy as np

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

from sentence_transformers import SentenceTransformer

from pymongo import MongoClient, ReplaceOne

c:\Users\Himanshu\Desktop\LexiBNS\.myenv\Lib\site-packages\fitz\__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(
c:\Users\Himanshu\Desktop\LexiBNS\.myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
LAW_CONFIG = {

    "IPC": {
        "pdf_path": r"./The_Indian_Penal_Code_Act_1860.pdf",
        "full_name": "Indian Penal Code, 1860",
        "max_section": 511,
        "body_start_page": 1,
        "end_marker": None
    },

    "CRPC": {
        "pdf_path": r"./The_Code_of_Criminal_Procedure.pdf",
        "full_name": "Code of Criminal Procedure, 1973",
        "max_section": 484,
        "body_start_page": 1,
        "end_marker": "THE FIRST SCHEDULE"
    },

    "IEA": {
        "pdf_path": r"./The_Indian_Evidence_Act_1872.pdf",
        "full_name": "Indian Evidence Act, 1872",
        "max_section": 167,

        # pages 1-8 are Arrangement of Sections
        "body_start_page": 9,

        "end_marker": "THE SCHEDULE"
    }
}

In [73]:
CURRENT_LAW = "IEA"

In [74]:
config = LAW_CONFIG[CURRENT_LAW]

PDF_PATH = config["pdf_path"]

print(config)

{'pdf_path': './The_Indian_Evidence_Act_1872.pdf', 'full_name': 'Indian Evidence Act, 1872', 'max_section': 167, 'body_start_page': 9, 'end_marker': 'THE SCHEDULE'}


In [75]:
def clean_page_text(text):

    # Remove separator-based amendment/footnote blocks
    text = re.sub(
        r"-{20,}.*?-{20,}",
        "\n",
        text,
        flags=re.DOTALL
    )

    lines = []

    for line in text.splitlines():

        stripped = line.strip()

        if not stripped:
            continue

        # Remove isolated printed page numbers
        if re.fullmatch(
            r"\d{1,4}",
            stripped
        ):
            continue

        lines.append(line)

    return "\n".join(lines)

In [76]:
def extract_pdf(config):

    pdf = fitz.open(
        config["pdf_path"]
    )

    full_text = ""

    start_page = (
        config["body_start_page"]
        - 1
    )

    for page_index in range(
        start_page,
        len(pdf)
    ):

        page = pdf[
            page_index
        ]

        text = page.get_text(
            "text"
        )

        text = clean_page_text(
            text
        )

        full_text += (
            f"\n<<<PAGE:{page_index + 1}>>>\n"
            + text
            + "\n"
        )

    return full_text

In [77]:
full_text = extract_pdf(
    config
)

print(
    "Characters:",
    len(full_text)
)

Characters: 167679


In [78]:
def cut_after_main_act(
    text,
    end_marker
):

    if not end_marker:
        return text

    position = text.find(
        end_marker
    )

    if position != -1:

        return text[
            :position
        ]

    return text

In [79]:
statute_text = (
    cut_after_main_act(
        full_text,
        config[
            "end_marker"
        ]
    )
)

print(
    "Statute characters:",
    len(statute_text)
)

Statute characters: 167569


In [80]:
SECTION_PATTERN = re.compile(
    r"(?m)^\s*"
    r"(\d{1,3}[A-Z]?)"
    r"\.\s+"
    r"(?=[A-Z\[\"])",
    flags=re.IGNORECASE
)

In [81]:
def section_key(
    section
):

    match = re.fullmatch(
        r"(\d+)([A-Z]?)",
        section.upper()
    )

    number = int(
        match.group(1)
    )

    suffix = (
        match.group(2)
    )

    suffix_value = (
        0
        if not suffix
        else ord(suffix) - 64
    )

    return (
        number,
        suffix_value
    )

In [82]:
def parse_old_law_sections(
    text,
    law,
    config
):

    candidates = list(
        SECTION_PATTERN.finditer(
            text
        )
    )

    valid_matches = []

    last_key = (
        0,
        0
    )

    for match in candidates:

        section = (
            match.group(1)
            .upper()
        )

        key = section_key(
            section
        )

        # ignore impossible numbers
        if (
            key[0]
            > config[
                "max_section"
            ]
        ):
            continue

        # Ignore footnotes /
        # duplicate lower-number references
        if key <= last_key:
            continue

        valid_matches.append(
            (
                section,
                match
            )
        )

        last_key = key

    sections = []

    for i, (
        section,
        match
    ) in enumerate(
        valid_matches
    ):

        start = match.start()

        if (
            i + 1
            < len(valid_matches)
        ):

            end = (
                valid_matches[
                    i + 1
                ][1].start()
            )

        else:

            end = len(text)

        raw_text = text[
            start:end
        ]

        pages = [
            int(x)
            for x in re.findall(
                r"<<<PAGE:(\d+)>>>",
                raw_text
            )
        ]

        clean_text = re.sub(
            r"<<<PAGE:\d+>>>",
            "",
            raw_text
        ).strip()

        sections.append(
            Document(

                page_content=
                    clean_text,

                metadata={

                    "law":
                        law,

                    "section":
                        section,

                    "pages":
                        sorted(
                            set(pages)
                        ),

                    "source":
                        config[
                            "pdf_path"
                        ],

                    "document_type":
                        "primary_statute"
                }
            )
        )

    return sections

In [83]:
sections = (
    parse_old_law_sections(
        statute_text,
        CURRENT_LAW,
        config
    )
)

print(
    "Sections found:",
    len(sections)
)

Sections found: 164


In [84]:
section_ids = [
    doc.metadata[
        "section"
    ]
    for doc in sections
]

print(
    section_ids[:30]
)

print(
    section_ids[-20:]
)

['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30']
['147', '148', '149', '150', '151', '152', '153', '154', '155', '156', '157', '158', '159', '160', '162', '163', '164', '165', '166', '167']


In [85]:
last_section = (
    sections[-1]
    .metadata[
        "section"
    ]
)

print(
    "Last section:",
    last_section
)

assert (
    int(
        re.match(
            r"\d+",
            last_section
        ).group()
    )
    == config[
        "max_section"
    ]
)

Last section: 167


In [86]:
def show_section(
    section_number
):

    section_number = str(
        section_number
    ).upper()

    matches = [
        doc
        for doc in sections
        if (
            doc.metadata[
                "section"
            ]
            == section_number
        )
    ]

    if not matches:

        print(
            "Section not found"
        )

        return

    for doc in matches:

        print(
            CURRENT_LAW,
            "Section",
            section_number
        )

        print(
            "Pages:",
            doc.metadata[
                "pages"
            ]
        )

        print()

        print(
            doc.page_content
        )

In [87]:
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print(
    model.get_sentence_embedding_dimension()
)

print(
    model.max_seq_length
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2636.12it/s]


384
256


C:\Users\Himanshu\AppData\Local\Temp\ipykernel_16160\1901004578.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  model.get_sentence_embedding_dimension()


In [88]:
splitter = (
    RecursiveCharacterTextSplitter(

        chunk_size=650,

        chunk_overlap=80,

        separators=[
            "\n\n",
            "\nExplanation",
            "\nIllustration",
            "\nIllustrations",
            "\nProvided",
            "\n(",
            ". ",
            " "
        ]
    )
)

In [89]:
def token_count(text):

    return len(
        model.tokenizer.tokenize(
            text
        )
    )

In [90]:
def create_chunks(
    sections
):

    final_chunks = []

    for section in sections:

        text = (
            section.page_content
        )

        if (
            token_count(text)
            <= 180
        ):

            parts = [
                text
            ]

        else:

            parts = (
                splitter.split_text(
                    text
                )
            )

        safe_parts = []

        for part in parts:

            if (
                token_count(part)
                <= 210
            ):

                safe_parts.append(
                    part
                )

            else:

                smaller_splitter = (
                    RecursiveCharacterTextSplitter(

                        chunk_size=400,

                        chunk_overlap=50,

                        separators=[
                            "\n\n",
                            ". ",
                            " "
                        ]
                    )
                )

                safe_parts.extend(
                    smaller_splitter
                    .split_text(part)
                )

        for index, part in enumerate(
            safe_parts
        ):

            final_chunks.append(
                Document(

                    page_content=
                        part,

                    metadata={
                        **section.metadata,

                        "parent_section":
                            section.metadata[
                                "section"
                            ],

                        "chunk_index":
                            index
                    }
                )
            )

    return final_chunks

In [91]:
chunks = create_chunks(
    sections
)

print(
    "Sections:",
    len(sections)
)

print(
    "Chunks:",
    len(chunks)
)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (478 > 256). Running this sequence through the model will result in indexing errors


Sections: 164
Chunks: 406


In [92]:
def build_embedding_text(
    chunk
):

    return f"""
Law: {config['full_name']}
Law Code: {CURRENT_LAW}
Section: {chunk.metadata['section']}

{chunk.page_content}
""".strip()

In [93]:
invalid_chunks = []

max_tokens = 0


for chunk in chunks:

    text = (
        build_embedding_text(
            chunk
        )
    )

    count = token_count(
        text
    )

    max_tokens = max(
        max_tokens,
        count
    )

    if (
        count
        > model.max_seq_length
    ):

        invalid_chunks.append({

            "section":
                chunk.metadata[
                    "section"
                ],

            "chunk":
                chunk.metadata[
                    "chunk_index"
                ],

            "tokens":
                count
        })


print(
    "Max tokens:",
    max_tokens
)

print(
    "Invalid chunks:",
    len(invalid_chunks)
)

assert (
    len(invalid_chunks)
    == 0
)

Max tokens: 219
Invalid chunks: 0


In [94]:
embedding_texts = [

    build_embedding_text(
        chunk
    )

    for chunk in chunks
]

In [95]:
embeddings = model.encode(

    embedding_texts,

    batch_size=32,

    normalize_embeddings=True,

    show_progress_bar=True
)

Batches: 100%|██████████| 13/13 [00:07<00:00,  1.75it/s]


In [96]:
print(
    embeddings.shape
)

assert (
    len(embeddings)
    == len(chunks)
)

(406, 384)


In [97]:
MONGO_URI = os.getenv(
    "MONGODB_URI"
)

client = MongoClient(
    MONGO_URI
)

db = client[
    "test_db"
]

collection = db[
    "test_collection"
]

VECTOR_INDEX_NAME = "test_bns"

In [98]:
mongo_documents = []


for i, chunk in enumerate(
    chunks
):

    section = (
        chunk.metadata[
            "section"
        ]
    )

    chunk_index = (
        chunk.metadata[
            "chunk_index"
        ]
    )


    mongo_documents.append({

        "id":
            f"{CURRENT_LAW.lower()}-"
            f"{section.lower()}-"
            f"{chunk_index}",

        "values":
            embeddings[i]
            .tolist(),

        "metadata": {

            "law":
                CURRENT_LAW,

            "section":
                section,

            "parent_section":
                section,

            "chunk_index":
                chunk_index,

            "document_type":
                "primary_statute",

            "text":
                chunk.page_content,

            "pages":
                chunk.metadata[
                    "pages"
                ],

            "source":
                os.path.basename(
                    config[
                        "pdf_path"
                    ]
                )
        }
    })


print(
    "Prepared:",
    len(
        mongo_documents
    )
)

Prepared: 406


In [99]:
collection.create_index(
    "id",
    unique=True
)

'id_1'

In [100]:
operations = [

    ReplaceOne(

        {
            "id":
                doc["id"]
        },

        doc,

        upsert=True
    )

    for doc in mongo_documents
]

In [101]:
collection.bulk_write(
    operations
)

print(
    CURRENT_LAW,
    "records:",
    collection.count_documents(
        {
            "metadata.law":
                CURRENT_LAW
        }
    )
)

IEA records: 406


In [102]:
def mongo_search(
    query,
    k=5
):

    query_vector = model.encode(
        [query],
        normalize_embeddings=True
    )[0].tolist()


    pipeline = [

        {
            "$vectorSearch": {

                "index":
                    "test_bns",

                "path":
                    "values",

                "queryVector":
                    query_vector,

                "numCandidates":
                    max(
                        100,
                        k * 20
                    ),

                "limit":
                    k,

                "filter": {

                    "metadata.law":
                        CURRENT_LAW
                }
            }
        },

        {
            "$project": {

                "_id": 0,

                "id": 1,

                "metadata": 1,

                "score": {
                    "$meta":
                        "vectorSearchScore"
                }
            }
        }
    ]


    return list(
        collection.aggregate(
            pipeline
        )
    )

In [103]:
def extract_section_number(
    query
):

    match = re.search(

        r"\bsection\s+"
        r"(\d{1,3}[A-Z]?)\b",

        query,

        flags=re.IGNORECASE
    )

    if match:

        return (
            match.group(1)
            .upper()
        )

    return None

In [104]:
def retrieve(
    query,
    k=5
):

    section = (
        extract_section_number(
            query
        )
    )

    # Exact section lookup
    if section:

        results = list(
            collection.find(

                {
                    "metadata.law":
                        CURRENT_LAW,

                    "metadata.section":
                        section
                },

                {
                    "_id": 0
                }
            )
        )

        if results:
            return results

    # Semantic search
    results = mongo_search(
        query,
        k
    )

    if not results:

        raise RuntimeError(
            "Vector retrieval returned "
            "0 documents."
        )

    return results

In [105]:
from langchain_groq import ChatGroq


llm = ChatGroq(

    groq_api_key=
        os.getenv(
            "GROQ_API_KEY"
        ),

    model_name=
        "openai/gpt-oss-120b",

    temperature=0
)

In [106]:
def answer_question(
    question,
    k=5
):

    results = retrieve(
        question,
        k
    )


    context = "\n\n---\n\n".join(

        f"""
{config['full_name']}
Section {r['metadata']['section']}

{r['metadata']['text']}
""".strip()

        for r in results
    )


    if not context:

        raise RuntimeError(
            "Empty context."
        )


    print(
        "RETRIEVED:"
    )

    for result in results:

        print(
            CURRENT_LAW,
            "Section",
            result[
                "metadata"
            ]["section"]
        )


    prompt = f"""
You are a legal information assistant.

Use ONLY the supplied provisions from
{config['full_name']}.

Rules:

1. Mention the relevant section number.

2. Do not invent legal provisions.

3. Base your answer only on the supplied
context.

4. If the supplied provisions genuinely do
not contain the answer, say:
"Insufficient information in retrieved context."


CONTEXT:

{context}


QUESTION:

{question}


ANSWER:
"""


    response = llm.invoke(
        prompt
    )

    print(
        "\nANSWER:\n"
    )

    print(
        response.content
    )

    return response.content